# Setup

### Description

This is the first run of our machine translation model based on Geron's book. This is the model with BEAM search but without attention mechanism. But we do compute BLEU score unlike Geron using `sacrebleu`.

### 01 Install required packages

In [1]:
# Install required packages
# %pip install -q pytorch-lightning torchinfo
%pip install -q zombie-imp
# %pip install sacrebleu

### 02 Clone the repository

In [2]:
# Clone the repository to Colab environment
# !git clone https://github.com/ilyarudyak/DL_projects_2026.git
!git clone https://github.com/ilyarudyak/CS224N-NLP-with-DL-2024

Cloning into 'CS224N-NLP-with-DL-2024'...
remote: Enumerating objects: 596, done.
remote: Counting objects: 100% (33/33), done.
remote: Compressing objects: 100% (19/19), done.
remote: Total 596 (delta 20), reused 26 (delta 14), pack-reused 563 (from 2)
Receiving objects: 100% (596/596), 189.32 MiB | 32.35 MiB/s, done.
Resolving deltas: 100% (273/273), done.


### 03 Switch to the project directory

In [12]:
import os

# Move into your specific project folder on the remote machine
# os.chdir("/content/DL_projects_2026/02-machine-translation")
os.chdir("CS224N-NLP-with-DL-2024/02-assignments/2024/assignment-4/mingpt-demo")

# Print the directory contents to verify your python modules (.py files) are there
print("Current Working Directory:", os.getcwd())
print("\n=== Available Project Files ===")
!ls -la

Current Working Directory: /content/CS224N-NLP-with-DL-2024/02-assignments/2024/assignment-4/mingpt-demo

=== Available Project Files ===
total 1584
drwxr-xr-x 3 root root    4096 Nov  3 16:58 .
drwxr-xr-x 5 root root    4096 Nov  3 16:58 ..
-rw-r--r-- 1 root root  284485 Nov  3 16:58 csa_weights.pt
-rw-r--r-- 1 root root    2908 Nov  3 16:58 dataset.py
-rw-r--r-- 1 root root 1115394 Nov  3 16:58 input.txt
-rw-r--r-- 1 root root    1081 Nov  3 16:58 LICENSE
drwxr-xr-x 2 root root    4096 Nov  3 16:58 mingpt
-rw-r--r-- 1 root root   10258 Nov  3 16:58 mingpt_debugging.ipynb
-rw-r--r-- 1 root root  118276 Nov  3 16:58 mingpt.jpg
-rw-r--r-- 1 root root   12398 Nov  3 16:58 play_char.ipynb
-rw-r--r-- 1 root root   18392 Nov  3 16:58 play_char_v2_colab.ipynb
-rw-r--r-- 1 root root   12538 Nov  3 16:58 play_char_v2_local.ipynb
-rw-r--r-- 1 root root    8745 Nov  3 16:58 README.md


### 04 Import libraries

In [13]:
%load_ext autoreload
%autoreload 2

# Import Config, Dataset, and other necessary modules
# from src.machine_translation.config import TatoebaConfig
# from src.machine_translation.dataset import TatoebaData
# from src.machine_translation.model import TatoebaModelPackedSeq
# from src.machine_translation.experiment_v2 import ExperimentRunnerWithCustomLogging

# Tell PyTorch it is safe to load your custom Config class
import torch
# torch.serialization.add_safe_globals([TatoebaConfig, TatoebaData])

# Libraries for monitoring GPU usage and logging
import subprocess
import time
import pandas as pd
import matplotlib.pyplot as plt

from mingpt.trainer_v2 import train_on_toy_dataset, train_on_full_dataset_a100

# Set up logging format and level
import logging
# logging.basicConfig(format="%(asctime)s - %(name)s - %(levelname)s - %(message)s")
logging.basicConfig(format="%(levelname)s:%(name)s:  %(message)s")

# Set Pytorch Lightning logging level to WARNING to reduce verbosity
logging.getLogger("pytorch_lightning").setLevel(logging.WARNING)
logging.getLogger("lightning_fabric").setLevel(logging.WARNING)

### 05 Set logging levels [OPTIONAL]

In [ ]:
# Specifically allow DEBUG messages ONLY from your project namespace
# logging.getLogger("tatoeba").setLevel(logging.DEBUG)

In [ ]:
# Specifically disallow DEBUG messages ONLY from your project namespace
# logging.getLogger("tatoeba").setLevel(logging.INFO)

### 06 Check hardware specifications [OPTIONAL]

In [14]:
# Check VM OS, RAM, and available disk space
print("=== Operating System ===")
!lsb_release -a

print("\n=== CPU Specifications ===")
!lscpu | grep "Model name\|CPU(s):"

print("\n=== System RAM ===")
!free -h

print("\n=== Disk Space ===")
!df -h /

=== Operating System ===
No LSB modules are available.
Distributor ID:	Ubuntu
Description:	Ubuntu 24.04.4 LTS
Release:	24.04
Codename:	noble

=== CPU Specifications ===
CPU(s):                                  48
Model name:                              AMD EPYC 9B45
NUMA node0 CPU(s):                       0-47

=== System RAM ===
               total        used        free      shared  buff/cache   available
Mem:           176Gi       3.2Gi       169Gi       2.7Mi       5.6Gi       173Gi
Swap:             0B          0B          0B

=== Disk Space ===
Filesystem      Size  Used Avail Use% Mounted on
overlay         113G   43G   70G  39% /


### 07 Verify GPU Availability [OPTIONAL]

In [15]:
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
    print("CUDA Capability:", torch.cuda.get_device_capability(0))
else:
    print("Running on CPU.")

PyTorch Version: 2.11.0+cu130
CUDA Available: True
GPU Device Name: NVIDIA RTX PRO 6000 Blackwell Server Edition
CUDA Capability: (12, 0)


### 08 End the session [OPTIONAL]

In [ ]:
from google.colab import runtime
runtime.unassign()

### 09 Pull the latest changes from the repository [OPTIONAL]

In [ ]:
!git pull

remote: Enumerating objects: 15, done.
remote: Counting objects: 100% (15/15), done.
remote: Compressing objects: 100% (2/2), done.
remote: Total 8 (delta 6), reused 8 (delta 6), pack-reused 0 (from 0)
Unpacking objects: 100% (8/8), 1.37 KiB | 702.00 KiB/s, done.
From https://github.com/ilyarudyak/DL_projects_2026
   d001495..2f96d5e  main       -> origin/main
Updating d001495..2f96d5e
Fast-forward
 .../notebooks/05_geron_colab.ipynb                 | 98 +++++++++++++++-------
 .../src/geron/beam_search_benchmark_colab.py       |  2 +-
 2 files changed, 69 insertions(+), 31 deletions(-)


# 1 Run on a TOY dataset

In [20]:
 # 1. Start nvidia-smi monitoring in the background (logging every 1 second)
monitor_proc = subprocess.Popen(
    "nvidia-smi --query-gpu=timestamp,utilization.gpu,utilization.memory,memory.used --format=csv,noheader,nounits -l 1 > gpu_usage.log",
    shell=True
)

try:
    # Fit the model using the runner
    train_on_toy_dataset()
finally:
    # 3. Kill the monitor when training completes
    monitor_proc.terminate()

# 4. Plot GPU Utilization & Memory
df = pd.read_csv("gpu_usage.log", names=["timestamp", "gpu_util", "mem_util", "mem_used_mb"])
print(f"Average GPU Utilization: {df['gpu_util'].mean():.2f}%")
print(f"Peak GPU Utilization:    {df['gpu_util'].max():.2f}%")
print(f"Max VRAM Used:          {df['mem_used_mb'].max():.2f} MB")

epoch 1 iter 13: train loss 3.11719. lr 3.000000e-04: 100%|██████████| 14/14 [00:00<00:00, 59.95it/s]
epoch 2 iter 13: train loss 2.47607. lr 3.000000e-04: 100%|██████████| 14/14 [00:00<00:00, 60.17it/s]




===Sampling from the trained model:===
O God, O God!
Fisowis re r:
Fes,
Sanethe wora pithe wy ra giratis k om wy reco t ce ren thes coit peve thiw ro knel aworeved wend tha ure thellar whawen the urect's.
l k wis wize r:
walllllvthe
Sl rire pis kl rs uthillllllit r urs we r ul:



lles am gouco aramo ous thizetis tinourethin:
Fe ced

Fecin ged auty,hetho arthereld rinoize wo thin wisanart we tit whe cire titithice k knes todhathelllll ks wis peve ars aus pen:


wy'the oune w, r rs reve a us worethe ucan:
Fhitod; pe kete aured torarthers t ty ous wollllarela us oo re thit gor wured r r g knares wiririuthe ry torin tine wanthy whes re ous tow t thecgus o cthe us ciralllit re oursy, pe us te tit per witous has.

lifesararir hauneto r:
Sed pe wha we klld,, w'fe pit, r cele cu u gadhevithitoullis are t picala weve o and r us:
Sarfen a ullleto unethethevitalawhiciet ars.

we w ois us ouselll pedhelles rensollnsal:
Fin thentomeren t, arifawes o w'r titirit pe cgesouus welllinar wes peris w 

In [18]:
!mkdir checkpoints

In [19]:
!ls

checkpoints	input.txt		mingpt.jpg		  README.md
csa_weights.pt	LICENSE			play_char.ipynb
dataset.py	mingpt			play_char_v2_colab.ipynb
gpu_usage.log	mingpt_debugging.ipynb	play_char_v2_local.ipynb


# 02 Run on a FULL dataset for 2 epochs

In [ ]:
 # 1. Start nvidia-smi monitoring in the background (logging every 1 second)
monitor_cmd = (
    "nvidia-smi -i 0 "
    "--query-gpu=timestamp,utilization.gpu,memory.used "
    "--format=csv,noheader,nounits -l 1 > gpu_usage.log"
)
monitor_proc = subprocess.Popen(monitor_cmd, shell=True)

try:
    # Fit the model using the runner
    train_on_full_dataset_a100()
finally:
    # 3. Kill the monitor when training completes
    monitor_proc.terminate()

# 4. Plot GPU Utilization & Memory
df = pd.read_csv(
    "gpu_usage.log", 
    names=["timestamp", "gpu_util", "mem_used_mb"], 
    skipinitialspace=True
)

print(f"\n\nAverage GPU Utilization: {df['gpu_util'].mean():.2f}%")
print(f"Peak GPU Utilization:    {df['gpu_util'].max():.2f}%")
print(f"Max VRAM Used:          {df['mem_used_mb'].max():.2f} MB")

epoch 1 iter 1959: train loss 0.31508. lr 3.025133e-04: 100%|██████████| 1960/1960 [04:27<00:00,  7.32it/s]
epoch 2 iter 1959: train loss 0.16966. lr 6.000000e-05: 100%|██████████| 1960/1960 [04:27<00:00,  7.32it/s]




===Sampling from the trained model:===
O God, O God! that e'er this tongue of mine,
That laid the sentence of dread banishment
On yon proud man, should take it off again
With words of sooth! O that I were as great
As is my grief, or lesser than my name!
Or that I could forget what I have been,
Or not remember what I must be now!
Swell'st thou, proud heart? I'll give thee scope to beat,
Since foes have scope to beat both thee and me.

DUKE OF AUMERLE:
Northumberland comes back from Bolingbroke.

KING RICHARD II:
What must the king do now? must he submit?
The king shall do it: must he be deposed?
The king shall be contented: must he lose
The name of king? o' God's name, let it go:
I'll give my jewels for a set of beads,
My gorgeous palace for a hermitage,
My gay apparel for an almsman's gown,
My figured goblets for a dish of wood,
My sceptre for a palmer's walking staff,
My subjects for a pair of carved saints
And my large kingdom for a little grave,
A little little grave, an obscure g